# 微信解密数据库探索

本 Notebook 用于浏览解密后的 SQLite 数据库，帮助你了解数据结构后再做数据治理。

**主要数据库：**
| 目录 | 说明 |
|------|------|
| `contact/` | 联系人、群聊成员 |
| `session/` | 会话列表（最近消息摘要） |
| `message/` | 聊天记录（按聊天对象分表 `Msg_<md5>`） |
| `sns/` | 朋友圈 |
| `favorite/` | 收藏 |
| 其他 | bizchat、emoticon、general 等 |

**章节导航：** §1 库概览 → §2 联系人（§2.0 字段 · §2.1 标签/群成员 · §2.2 extra_buffer）→ §3 会话 → §4 聊天记录 → §5/§6 其他库


In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib
import os
import sqlite3

import pandas as pd
import zstandard as zstd
from IPython.display import display as ipy_display

# 解密数据库根目录（可按需修改）
DECRYPTED_DIR = Path("../wechat-decrypt-personal/decrypted")

CONTACT_DB = DECRYPTED_DIR / "contact" / "contact.db"
SESSION_DB = DECRYPTED_DIR / "session" / "session.db"
MESSAGE_DIR = DECRYPTED_DIR / "message"

assert DECRYPTED_DIR.is_dir(), f"目录不存在: {DECRYPTED_DIR}"
print(f"数据目录: {DECRYPTED_DIR}")


In [ ]:
import re

_zstd = zstd.ZstdDecompressor()

MSG_TYPE_NAMES = {
    1: "文本", 3: "图片", 34: "语音", 42: "名片", 43: "视频",
    47: "表情", 48: "位置", 49: "链接/文件", 50: "通话",
    10000: "系统", 10002: "撤回",
}


def connect_ro(db_path: Path) -> sqlite3.Connection:
    """只读连接；immutable=1 绕过 WSL 访问 Windows 盘时 WAL 锁冲突，且不生成 -wal/-shm。"""
    return sqlite3.connect(f"file:{db_path}?mode=ro&immutable=1", uri=True)


def split_msg_type(t):
    try:
        t = int(t)
    except (TypeError, ValueError):
        return 0, 0
    if t > 0xFFFFFFFF:
        return t & 0xFFFFFFFF, t >> 32
    return t, 0


def format_msg_type(t):
    base, _ = split_msg_type(t)
    return MSG_TYPE_NAMES.get(base, f"type={t}")


def decode_content(raw, ct_flag=0) -> str:
    if raw is None:
        return ""
    if isinstance(raw, bytes):
        if ct_flag == 4:
            try:
                return _zstd.decompress(raw).decode("utf-8", errors="replace")
            except Exception:
                return "[zstd 解压失败]"
        return raw.decode("utf-8", errors="replace")
    return str(raw)


def xml_extract(content: str, *tags) -> str:
    for tag in tags:
        m = re.search(rf"<{tag}>(.*?)</{tag}>", content, re.DOTALL)
        if m:
            return m.group(1).strip()
    return ""


def friendly_content(msg_type, content: str, chat_username: str = "", sender_username: str = "") -> str:
    """把原始 message_content 转成可读摘要。"""
    base, _ = split_msg_type(msg_type)
    text = content or ""
    is_group = chat_username.endswith("@chatroom") or chat_username.endswith("@openim")
    if is_group and ":\n" in text:
        text = text.split(":\n", 1)[1]

    if base == 1:
        return text
    if base == 3:
        return "[图片]"
    if base == 34:
        return "[语音]"
    if base == 42:
        return f"[名片: {xml_extract(content, 'nickname') or '?'}]"
    if base == 43:
        return "[视频]"
    if base == 47:
        return "[表情]"
    if base == 48:
        return f"[位置: {xml_extract(content, 'label') or '?'}]"
    if base == 49:
        title = xml_extract(content, "title")
        desc = xml_extract(content, "des")
        url = xml_extract(content, "url")
        lines = [f"[分享: {title}]" if title else "[链接/文件]"]
        if desc:
            lines.append(f"摘要: {desc[:120]}")
        if url:
            lines.append(f"链接: {url}")
        return "\n".join(lines)
    if base in (10000, 10002):
        return f"[系统] {text[:300]}"
    return text[:500]


def display_name(row) -> str:
    remark, nick, username = row.get("remark", ""), row.get("nick_name", ""), row.get("username", "")
    return remark or nick or username


def username_to_msg_table(username: str) -> str:
    return f"Msg_{hashlib.md5(username.encode()).hexdigest()}"


## 1. 数据库文件概览

扫描 `decrypted/` 下所有 `.db` 文件，按大小排序。


In [ ]:
db_files = sorted(DECRYPTED_DIR.rglob("*.db"))
rows = []
for p in db_files:
    rel = p.relative_to(DECRYPTED_DIR)
    size_mb = p.stat().st_size / 1024 / 1024
    rows.append({"相对路径": str(rel), "大小(MB)": round(size_mb, 2)})

db_overview = pd.DataFrame(rows).sort_values("大小(MB)", ascending=False).reset_index(drop=True)
print(f"共 {len(db_overview)} 个数据库，合计 {db_overview['大小(MB)'].sum():.1f} MB")
db_overview


In [ ]:
def list_tables(db_path: Path, skip_msg_tables: bool = True) -> pd.DataFrame:
    """列出单个库中的表、列名、行数。message 库的 Msg_* 表默认跳过 COUNT（数量多且慢）。"""
    conn = connect_ro(db_path)
    tables = [
        r[0] for r in conn.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name"
        )
    ]
    rows = []
    for name in tables:
        if skip_msg_tables and name.startswith("Msg_"):
            cols = [c[1] for c in conn.execute(f"PRAGMA table_info([{name}])")]
            rows.append({"表名": name, "行数": "(Msg_* 跳过)", "列": ", ".join(cols[:8]) + ("..." if len(cols) > 8 else "")})
            continue
        cols = [c[1] for c in conn.execute(f"PRAGMA table_info([{name}])")]
        try:
            cnt = conn.execute(f"SELECT COUNT(*) FROM [{name}]").fetchone()[0]
        except Exception as e:
            cnt = f"ERR: {e}"
        rows.append({"表名": name, "行数": cnt, "列": ", ".join(cols[:8]) + ("..." if len(cols) > 8 else "")})
    conn.close()
    return pd.DataFrame(rows)


# 查看核心库结构（修改 db_rel 可切换其他库）
db_rel = "contact/contact.db"
list_tables(DECRYPTED_DIR / db_rel)


## 2. 联系人 (contact.db)

`contact` 表包含好友、群聊、公众号、群成员等所有可见账号；显示名优先取 `remark`，其次 `nick_name`。

- **§2.0** 全库字段说明 + 示例好友A contact 行示例
- **§2.1** `contact_label` 与 `chatroom_member`
- **§2.2** `extra_buffer` Protobuf 扩展包（标签、签名、地区、朋友圈封面等）


### 2.0 contact.db 表结构说明

`contact/contact.db` 存储**所有可见账号**的信息：好友、群聊、群成员、公众号、企业微信联系人等。
注意：这里的「联系人」≠ 仅好友列表，还包含群里的陌生人、已删除但未 purge 的记录等。

同目录还有 `contact_fts.db`（全文搜索索引），治理时通常以 `contact.db` 为准。

---

#### 表一览（本库共 15 张业务表）

| 表名 | 约行数 | 作用 |
|------|--------|------|
| **contact** | 3.2 万 | 核心表：每个账号一条（含群成员 exploded 行） |
| **name2id** | 3.2 万 | 所有 username 的去重索引 |
| **chatroom_member** | 3.9 万 | 群 ID ↔ 成员 contact.id 的多对多关系 |
| **chat_room** | 586 | 群聊元数据（群主等） |
| **chat_room_info_detail** | 582 | 群公告、群状态 |
| **biz_info** | 445 | 公众号/服务号扩展信息 |
| **contact_label** | 27 | 好友标签（标签 ID ↔ 名称） |
| **ticket_info** | 145 | 内部 ticket 缓存（用途未完全公开） |
| **openim_*** | 少量 | 企业微信/OpenIM 文案与账号类型 |
| **stranger / oplog / encrypt_name2id** | 0 | 结构保留，当前无数据 |

---

#### `contact` 表字段（22 列，最常用）

| 字段 | 类型 | 含义 |
|------|------|------|
| **id** | INTEGER，主键 | 本地 contact 行 ID；`chatroom_member.member_id` 引用此值 |
| **username** | TEXT | 稳定账号 ID：`wxid_*`、`*@chatroom`、`*@openim`、`gh_*`（公众号）等 |
| **local_type** | INTEGER | 联系人类别（见下表）；治理时常过滤 `local_type=3` 去掉群成员冗余行 |
| **alias** | TEXT | 微信号（用户自设 ID，如 `georgezxm`），可为空 |
| **encrypt_username** | TEXT | 加密/陌生人场景下的 username 变体（含 `@stranger` 后缀） |
| **flag** | INTEGER | 微信内部状态位（好友关系、星标、黑名单等 bitmask，未完全公开） |
| **delete_flag** | INTEGER | 是否已删除：`0`=有效，`1`=已删（本库仅 4 条 delete_flag=1） |
| **verify_flag** | INTEGER | 验证/好友请求相关标志 |
| **remark** | TEXT | **备注名**（你给对方设的备注）；显示名优先取此字段 |
| **remark_quan_pin** | TEXT | 备注名全拼（搜索用，如 `zhouxumingjiushao`） |
| **remark_pin_yin_initial** | TEXT | 备注名拼音首字母（如 `ZXMJS`） |
| **nick_name** | TEXT | **微信昵称**（对方自己设的名字） |
| **pin_yin_initial** | TEXT | 昵称拼音首字母 |
| **quan_pin** | TEXT | 昵称全拼 |
| **big_head_url** | TEXT | 大头像 CDN URL |
| **small_head_url** | TEXT | 小头像 CDN URL |
| **head_img_md5** | TEXT | 头像 MD5，用于缓存校验 |
| **chat_room_notify** | INTEGER | 群消息免打扰：`0`=正常通知，`1`=免打扰 |
| **is_in_chat_room** | INTEGER | 是否仍在群内（退群/被踢后可能变化） |
| **description** | TEXT | 好友描述/签名类文本（如「十一学校玉泉路校区，政治老师」） |
| **extra_buffer** | BLOB | protobuf 扩展包（详见 **§2.2**：标签 #30、签名 #4、地区 #5-7、封面 #27 等） |
| **chat_room_type** | INTEGER | 群类型标记；非群聊多为 `0`，部分群为 `2` |

**`local_type` 常见取值（微信 4.x 实测）：**

| 值 | 含义 | 示例 |
|----|------|------|
| 0 | 信息不完整的联系人/陌生人 | 仅有 nick_name，无 remark |
| 1 | **好友**或**少量特殊账号** | 有 remark 的好友；部分 `@chatroom` |
| 2 | **群聊实体**（`xxx@chatroom`） | 群名在 nick_name，备注在 remark |
| 3 | **群成员**（每个群每个成员一条） | 数量最多；导出联系人时常 `WHERE local_type != 3` |
| 5 | 企业微信 / OpenIM 联系人 | `2598498...@openim` |
| 6 | OpenIM 变体 | 同上，部分无 nick_name |

**显示名规则（本项目统一）：** `remark` → `nick_name` → `username`

---

#### `chat_room` 表字段

| 字段 | 含义 |
|------|------|
| **id** | 群主表主键；与 `chatroom_member.room_id` 关联 |
| **username** | 群 ID（`*@chatroom`） |
| **owner** | 群主 wxid |
| **ext_buffer** | 群扩展信息（protobuf） |

#### `chat_room_info_detail` 表字段

| 字段 | 含义 |
|------|------|
| **room_id_** | 主键，对应 chat_room.id |
| **username_** | 群 ID |
| **announcement_** | 群公告纯文本 |
| **announcement_editor_** | 公告发布者 wxid |
| **announcement_publish_time_** | 公告发布时间（Unix 秒） |
| **chat_room_status_** | 群状态标志 |
| **xml_announcement_** | 公告 XML 原文 |
| **ext_buffer_** | 扩展二进制 |

#### `chatroom_member` 表字段

| 字段 | 含义 |
|------|------|
| **room_id** | 群 ID（`chat_room.id`） |
| **member_id** | 成员 ID（`contact.id`） |

#### `contact_label` 表字段

| 字段 | 含义 |
|------|------|
| **label_id_** | 标签 ID |
| **label_name_** | 标签名称（如「北大地空」「清华」） |
| **sort_order_** | 排序序号 |

#### `biz_info` 表字段（公众号/服务号）

| 字段 | 含义 |
|------|------|
| **id** | 主键 |
| **username** | 公众号 ID（`gh_*`） |
| **type / child_type** | 账号类型 |
| **accept_type** | 消息接收类型 |
| **version** | 版本号 |
| **external_info / brand_info** | 品牌/扩展 JSON 或 XML |
| **brand_icon_url** | 公众号图标 URL |
| **brand_list / brand_flag** | 品牌列表与标志 |
| **belong** | 归属信息 |
| **ext_buffer** | 扩展二进制 |
| **home_url** | 主页链接 |
| **sync_version** | 同步版本 |

#### 其他表

| 表 | 字段 | 含义 |
|----|------|------|
| **name2id** | username | 全部 username 主键索引 |
| **encrypt_name2id** | username | 加密 username 索引（当前空） |
| **ticket_info** | id, ticket | 内部 ticket 字符串 |
| **stranger** | 同 contact 结构 | 陌生人缓存（当前空） |
| **stranger_ticket_info** | id, ticket | 陌生人 ticket（当前空） |
| **oplog** | id, buffer | 操作日志 blob（当前空） |
| **openim_acct_type** | lang_id, acc_type_id, update_time, ext_buffer | OpenIM 账号类型文案 |
| **openim_appid** | lang_id, app_id, acct_type_id, ... | OpenIM 应用 ID 映射 |
| **openim_wording** | app_id, wording_id, wording, pinyin, ... | OpenIM 界面文案 |

---

下方 **§2.0 示例** 以 **示例好友A** 的 `contact` 行为例，展示各字段真实取值。


#### 2.0 实际示例：示例好友A的 contact 行

运行下一单元格可查看该联系人在 `contact` 表中的**全部 22 个字段**及可读解释。


In [ ]:
# contact.db 全表概览 + 单条联系人字段展开
CONTACT_EXAMPLE = "wxid_friend_example"  # 示例好友A

conn = connect_ro(CONTACT_DB)

# 各表行数
table_stats = []
for name, in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name"
):
    cnt = conn.execute(f"SELECT COUNT(*) FROM [{name}]").fetchone()[0]
    table_stats.append({"表名": name, "行数": cnt})
contact_tables_df = pd.DataFrame(table_stats)
print("contact.db 各表行数：")
contact_tables_df


In [ ]:
# 展开示例好友A的全部 contact 字段
CONTACT_EXAMPLE = "wxid_friend_example"  # 示例好友A

FIELD_DOCS = {
    "id": "本地 contact 行 ID",
    "username": "稳定账号 ID（wxid）",
    "local_type": "联系人类别：1=好友",
    "alias": "微信号（自设 ID）",
    "encrypt_username": "加密 username / 陌生人变体",
    "flag": "内部状态位（好友/星标等）",
    "delete_flag": "0=有效，1=已删除",
    "verify_flag": "验证/好友请求标志",
    "remark": "备注名（显示名优先）",
    "remark_quan_pin": "备注全拼",
    "remark_pin_yin_initial": "备注拼音首字母",
    "nick_name": "微信昵称",
    "pin_yin_initial": "昵称拼音首字母",
    "quan_pin": "昵称全拼",
    "big_head_url": "大头像 URL",
    "small_head_url": "小头像 URL",
    "head_img_md5": "头像 MD5",
    "chat_room_notify": "群免打扰：0=正常，1=免打扰",
    "is_in_chat_room": "是否仍在群内",
    "description": "好友描述/签名",
    "extra_buffer": "protobuf 扩展（标签、地区等）",
    "chat_room_type": "群类型；单聊为 0",
}

conn = connect_ro(CONTACT_DB)
cols = [c[1] for c in conn.execute("PRAGMA table_info(contact)")]
row = conn.execute(
    f"SELECT {', '.join('['+c+']' for c in cols)} FROM contact WHERE username=?",
    (CONTACT_EXAMPLE,),
).fetchone()
conn.close()

if not row:
    raise RuntimeError(f"contact 表中未找到 {CONTACT_EXAMPLE}")

def _fmt_val(col, val):
    if val is None:
        return ""
    if col in ("big_head_url", "small_head_url", "encrypt_username") and isinstance(val, str) and len(val) > 80:
        return val[:80] + "…"
    if col == "extra_buffer" and isinstance(val, bytes):
        return f"<BLOB {len(val)} bytes>"
    return val

contact_detail = pd.DataFrame({
    "字段": cols,
    "含义": [FIELD_DOCS.get(c, "") for c in cols],
    "值": [_fmt_val(c, v) for c, v in zip(cols, row)],
})

r = dict(zip(cols, row))
print(f"显示名: {r.get('remark') or r.get('nick_name') or r.get('username')}")
print(f"username: {r.get('username')}  |  local_type: {r.get('local_type')}  |  alias: {r.get('alias')}")
print(f"description: {r.get('description') or '(空)'}")
contact_detail


### 2.1 `contact_label` 与 `chatroom_member` 详解

这两张表都**不直接存显示名**，而是通过 ID 与其他表关联。

---

#### `contact_label` — 好友标签字典

| 字段 | 键类型 | 含义 |
|------|--------|------|
| **label_id_** | **主键（PK）** | 标签唯一 ID，如 `25` =「21科考」 |
| **label_name_** | 业务名 | 你在微信里设置的标签名称 |
| **sort_order_** | 排序 | 标签在微信里的显示顺序（从 0 起） |

**联系人 ↔ 标签怎么关联？** 没有独立的关联表。标签 ID 写在 `contact.extra_buffer`（protobuf）的 **Field #30** 里，格式为逗号分隔字符串，如 `"9,24"` 表示同时属于标签 9 和 24。

```
contact_label.label_id_  ←──  contact.extra_buffer (field 30: "9,24")
```

---

#### `chatroom_member` — 群成员关系表

| 字段 | 键类型 | 含义 |
|------|--------|------|
| **room_id** | 外键 | 指向 `chat_room.id`（不是 `@chatroom` 字符串） |
| **member_id** | 外键 | 指向 `contact.id`（不是 wxid） |

**联合唯一约束：** `UNIQUE(room_id, member_id)` — 同一人在同一群只出现一行。

```
chat_room.id  ←── chatroom_member.room_id
contact.id    ←── chatroom_member.member_id
chat_room.username = xxx@chatroom     contact.username = wxid_*
```

本库约 **3.9 万** 条成员关系，**582** 个群，平均每群 ~66 人（大群可达 500 上限）。

下方示例：**示例好友A** 的标签归属，以及其所在群 **示例群聊** 的成员列表。


In [ ]:
def extract_pb_field_30(data: bytes) -> str | None:
    """从 contact.extra_buffer 提取 Field #30（标签 ID 列表，逗号分隔）。"""
    if not data:
        return None
    pos, n = 0, len(data)
    while pos < n:
        tag = shift = 0
        while pos < n:
            b = data[pos]; pos += 1
            tag |= (b & 0x7f) << shift
            if not (b & 0x80):
                break
            shift += 7
        field_num, wire_type = tag >> 3, tag & 0x07
        if wire_type == 0:
            while pos < n and data[pos] & 0x80:
                pos += 1
            pos += 1
        elif wire_type == 2:
            length = shift = 0
            while pos < n:
                b = data[pos]; pos += 1
                length |= (b & 0x7f) << shift
                if not (b & 0x80):
                    break
                shift += 7
            if field_num == 30:
                try:
                    return data[pos:pos + length].decode("utf-8")
                except Exception:
                    return None
            pos += length
        elif wire_type == 1:
            pos += 8
        elif wire_type == 5:
            pos += 4
        else:
            break
    return None


conn = connect_ro(CONTACT_DB)

# 1) 全部标签
labels_df = pd.read_sql_query(
    "SELECT label_id_ AS label_id, label_name_ AS label_name, sort_order_ AS sort_order FROM contact_label ORDER BY sort_order_",
    conn,
)
print(f"共 {len(labels_df)} 个好友标签")
labels_df


In [ ]:
# 2) 扫描 extra_buffer，构建「标签 → 成员」映射
label_map = labels_df.set_index("label_id")["label_name"].to_dict()
tag_members: dict[int, list] = {lid: [] for lid in label_map}

for username, remark, nick, buf in conn.execute(
    "SELECT username, remark, nick_name, extra_buffer FROM contact WHERE extra_buffer IS NOT NULL"
):
    raw = extract_pb_field_30(buf)
    if not raw:
        continue
    display_name = remark or nick or username
    for lid_s in raw.split(","):
        try:
            lid = int(lid_s.strip())
        except ValueError:
            continue
        if lid in tag_members:
            tag_members[lid].append({"username": username, "display_name": display_name})

tag_stats = pd.DataFrame([
    {"label_id": lid, "label_name": label_map[lid], "成员数": len(members)}
    for lid, members in tag_members.items()
]).sort_values("成员数", ascending=False)
print("各标签成员数量（Top 10）：")
tag_stats.head(10)


In [ ]:
# 3) 示例：示例好友A属于哪些标签？
EXAMPLE_FRIEND_USERNAME = "wxid_friend_example"
row = conn.execute(
    "SELECT id, remark, nick_name, extra_buffer FROM contact WHERE username=?",
    (EXAMPLE_FRIEND_USERNAME,),
).fetchone()
contact_id, remark, nick, buf = row
label_raw = extract_pb_field_30(buf)
label_ids = [int(x) for x in label_raw.split(",")] if label_raw else []

zhou_tags = pd.DataFrame([
    {"label_id": lid, "label_name": label_map.get(lid, "?")}
    for lid in label_ids
])
print(f"{remark or nick}  contact.id={contact_id}")
print(f"extra_buffer Field#30 = {label_raw!r}")
zhou_tags

# 4) 示例：查看某个标签下的成员（默认「21科考」，可改 TAG_NAME）
TAG_NAME = "21科考"
tag_row = labels_df.loc[labels_df["label_name"] == TAG_NAME]
if not tag_row.empty:
    tid = int(tag_row.iloc[0]["label_id"])
    members = tag_members.get(tid, [])
    print(f"\n标签 [{TAG_NAME}]（label_id={tid}）共 {len(members)} 人，前 15 人：")
    pd.DataFrame(members).head(15)
else:
    print(f"未找到标签: {TAG_NAME}")


In [ ]:
# ── chatroom_member 示例 ──
EXAMPLE_ROOM_USERNAME = "1234567890@chatroom"  # 示例群聊（含示例好友A）

room = conn.execute(
    """
    SELECT cr.id AS room_id, cr.username, cr.owner,
           c.nick_name AS room_nick, c.remark AS room_remark
    FROM chat_room cr
    JOIN contact c ON c.username = cr.username
    WHERE cr.username = ?
    """,
    (EXAMPLE_ROOM_USERNAME,),
).fetchone()

if not room:
    raise RuntimeError(f"未找到群: {EXAMPLE_ROOM_USERNAME}")

room_id, room_username, owner, room_nick, room_remark = room
room_title = room_remark or room_nick or room_username
member_cnt = conn.execute(
    "SELECT COUNT(*) FROM chatroom_member WHERE room_id=?", (room_id,)
).fetchone()[0]

print(f"群: {room_title}")
print(f"  chat_room.id (room_id) = {room_id}")
print(f"  username = {room_username}")
print(f"  群主 owner = {owner}")
print(f"  成员数 = {member_cnt}\n")

members_df = pd.read_sql_query(
    """
    SELECT cm.room_id, cm.member_id,
           c.username, c.remark, c.nick_name, c.local_type
    FROM chatroom_member cm
    JOIN contact c ON c.id = cm.member_id
    WHERE cm.room_id = ?
    ORDER BY c.remark, c.nick_name
    """,
    conn,
    params=(room_id,),
)
members_df["display_name"] = members_df.apply(
    lambda r: r["remark"] or r["nick_name"] or r["username"], axis=1
)
members_df[["member_id", "username", "display_name", "local_type"]]


In [ ]:
# 5) 用 member_id 反查：示例好友A在该群中的关联行
zhou_in_room = members_df.loc[members_df["username"] == EXAMPLE_FRIEND_USERNAME]
print("示例好友A在本群的关系：")
print(f"  contact.id = member_id = {int(zhou_in_room.iloc[0]['member_id'])}")
print(f"  chat_room.id = room_id = {room_id}")
zhou_in_room[["room_id", "member_id", "username", "display_name"]]

# 6) 示例好友A所在的所有群（room_id 列表）
example_friend_groups = pd.read_sql_query(
    """
    SELECT cm.room_id, cr.username AS room_username,
           c.remark AS room_remark, c.nick_name AS room_nick,
           (SELECT COUNT(*) FROM chatroom_member x WHERE x.room_id = cm.room_id) AS member_count
    FROM chatroom_member cm
    JOIN contact mc ON mc.id = cm.member_id AND mc.username = ?
    JOIN chat_room cr ON cr.id = cm.room_id
    JOIN contact c ON c.username = cr.username
    ORDER BY member_count DESC
    LIMIT 15
    """,
    conn,
    params=(EXAMPLE_FRIEND_USERNAME,),
)
conn.close()
example_friend_groups["room_name"] = example_friend_groups.apply(
    lambda r: r["room_remark"] or r["room_nick"] or r["room_username"], axis=1
)
print(f"\n示例好友A共出现在 {len(example_friend_groups)} 个群（展示前 15 个）：")
example_friend_groups[["room_id", "room_name", "room_username", "member_count"]]


### 2.2 `contact.extra_buffer` 详解（Protobuf 扩展包）

是的：**每个联系人的好友标签**（以及签名、地区、朋友圈封面等）主要都存在 `contact.extra_buffer` 里，而不是单独的关联表。

`extra_buffer` 是 **Protocol Buffers（protobuf）** 序列化的 BLOB。微信没有公开 schema，以下字段号来自本库实测与 `wechat-decrypt` 项目逆向用法；带「推测」的字段请谨慎用于生产规则。

本库约 **3.0 万** 条 contact 有非空 `extra_buffer`。

#### 已确认 / 高置信字段

| Field # | 类型 | 含义 | 示例 |
|---------|------|------|------|
| **4** | string | **个性签名** | `我们一路奋战，不是为了改变世界…` |
| **5** | string | **国家/地区** | `CN` |
| **6** | string | **省份** | `Beijing` |
| **7** | string | **城市** | `Haidian` |
| **25** | string | **朋友圈背景 ID** | `918113684029440_918113684029440` |
| **27** | bytes | **朋友圈封面**（嵌套 protobuf） | 内含封面图 URL、时间戳等 |
| **30** | string | **好友标签 ID**（逗号分隔） | `9,24` → 标签A、标签B |

#### 常见但含义未完全公开的字节/整型字段

| Field # | 类型 | 推测含义 | 常见值 |
|---------|------|----------|--------|
| 2 | varint | 性别/账号属性 | `1` |
| 3 | varint | 内部标志 | `0` |
| 8 | varint | 添加来源/场景 | `0`~`14` |
| 11 | varint | 联系人类型 | `1`=好友, `3`=自己等 |
| 17, 18 | varint | 有效/展示标志 | `1` |
| 10, 38 | varint | 占位/全 1 位图 | `4294967295` |
| 14, 33 | bytes | 短二进制块 | 2 bytes |
| 15~16, 19~24, 26, 28~29, 31~32, 34~37 | 混合 | 多为空或固定默认值 | — |

#### Field #27 嵌套结构（朋友圈封面）

| 子 Field | 含义 |
|----------|------|
| **2** | 封面图 URL（`shmmsns.qpic.cn/...`） |
| **3** | 大整数时间戳/版本号 |
| **4, 5** | 数值参数（可能与尺寸/标志有关） |

#### 与 `description` 列的区别

| 列 | 内容 |
|----|------|
| `description` | 明文 TEXT，常是**你给对方写的好友描述/备注说明** |
| `extra_buffer` #4 | 对方的**微信个性签名**（对方自己设的） |

下方示例解析 **示例好友A** 的 `extra_buffer`，并与 **示例好友B**、**自己** 对比。


In [ ]:
def read_pb_varint(data: bytes, pos: int) -> tuple[int, int]:
    v = shift = 0
    while pos < len(data):
        b = data[pos]; pos += 1
        v |= (b & 0x7f) << shift
        if not (b & 0x80):
            break
        shift += 7
    return v, pos


def parse_protobuf(data: bytes) -> list[tuple[int, str, object]]:
    """解析 protobuf，返回 [(field_num, type, value), ...]"""
    if not data:
        return []
    rows, pos = [], 0
    while pos < len(data):
        tag, pos = read_pb_varint(data, pos)
        fn, wt = tag >> 3, tag & 7
        if wt == 0:
            v, pos = read_pb_varint(data, pos)
            rows.append((fn, "varint", v))
        elif wt == 2:
            ln, pos = read_pb_varint(data, pos)
            chunk = data[pos:pos + ln]; pos += ln
            try:
                s = chunk.decode("utf-8")
                printable = sum(1 for c in s if c.isprintable() or c in "\n\r\t")
                if len(s) > 0 and printable / len(s) > 0.85:
                    rows.append((fn, "string", s))
                else:
                    rows.append((fn, "bytes", chunk))
            except Exception:
                rows.append((fn, "bytes", chunk))
        elif wt == 1:
            pos += 8
        elif wt == 5:
            pos += 4
        else:
            break
    return rows


EXTRA_BUFFER_FIELD_DOCS = {
    2: ("性别/账号属性（推测）", "varint"),
    3: ("内部标志（推测）", "varint"),
    4: ("个性签名", "string"),
    5: ("国家/地区", "string"),
    6: ("省份", "string"),
    7: ("城市", "string"),
    8: ("添加来源/场景（推测）", "varint"),
    9: ("附加文本", "string"),
    10: ("位标志占位", "varint"),
    11: ("联系人类型（推测）", "varint"),
    25: ("朋友圈背景 ID", "string"),
    27: ("朋友圈封面（嵌套 protobuf）", "bytes"),
    30: ("好友标签 ID 列表", "string"),
    38: ("位标志占位", "varint"),
}


def decode_extra_buffer(buf: bytes, label_map: dict | None = None) -> pd.DataFrame:
    """把 extra_buffer 解析为可读 DataFrame。"""
    rows = []
    for fn, typ, val in parse_protobuf(buf):
        name, expected = EXTRA_BUFFER_FIELD_DOCS.get(fn, (f"Field #{fn}", typ))
        if typ == "bytes" and fn == 27:
            nested = parse_protobuf(val)
            url = next((c for a, b, c in nested if b == "string"), None)
            display = f"封面URL: {url[:70]}…" if url and len(url) > 70 else (f"封面URL: {url}" if url else f"<{len(val)} bytes>")
        elif typ == "bytes":
            display = f"<{len(val)} bytes>" if val else ""
        elif typ == "string" and fn == 30 and label_map:
            ids = [x.strip() for x in str(val).split(",") if x.strip()]
            names = [label_map.get(int(i), str(i)) for i in ids if i.isdigit()]
            display = f"{val} → {', '.join(names)}" if names else str(val)
        else:
            display = val
        if display in ("", 0, None, "<0 bytes>"):
            continue
        rows.append({"Field#": fn, "含义": name, "类型": typ, "值": display})
    return pd.DataFrame(rows)


conn = connect_ro(CONTACT_DB)
label_map = dict(conn.execute("SELECT label_id_, label_name_ FROM contact_label"))
buf_cnt = conn.execute(
    "SELECT COUNT(*) FROM contact WHERE extra_buffer IS NOT NULL AND length(extra_buffer) > 0"
).fetchone()[0]
print(f"有 extra_buffer 的 contact 行数: {buf_cnt:,}")


In [ ]:
# 示例 1：示例好友Aextra_buffer 全展开
EXAMPLE_FRIEND_USERNAME = "wxid_friend_example"
row = conn.execute(
    "SELECT remark, nick_name, description, extra_buffer FROM contact WHERE username=?",
    (EXAMPLE_FRIEND_USERNAME,),
).fetchone()
remark, nick, description, buf = row

print(f"联系人: {remark or nick}")
print(f"description 列（你写的好友描述）: {description or '(空)'}")
print(f"extra_buffer 大小: {len(buf)} bytes\n")

zhou_extra_df = decode_extra_buffer(buf, label_map)
zhou_extra_df


In [ ]:
# 示例 2：对比三人 —— 签名 / 地区 / 标签
COMPARE_USERS = [
    ("wxid_friend_example", "示例好友A"),
    ("wxid_friend_example_b", "示例好友B"),
    ("wxid_personal_owner", "自己"),
]

def pick_extra_fields(buf: bytes) -> dict:
    d = {fn: val for fn, typ, val in parse_protobuf(buf)}
    labels = d.get(30, "")
    label_names = ""
    if labels:
        label_names = ", ".join(
            label_map.get(int(x.strip()), x)
            for x in str(labels).split(",")
            if x.strip().isdigit()
        )
    region = " ".join(x for x in [d.get(5, ""), d.get(6, ""), d.get(7, "")] if x)
    return {
        "个性签名(#4)": d.get(4, ""),
        "地区(#5-7)": region or "(空)",
        "标签(#30)": label_names or labels or "(无)",
        "朋友圈背景(#25)": d.get(25, "") or "(无)",
    }

compare_rows = []
for username, title in COMPARE_USERS:
    buf = conn.execute(
        "SELECT extra_buffer FROM contact WHERE username=?", (username,)
    ).fetchone()[0]
    fields = pick_extra_fields(buf)
    fields["联系人"] = title
    compare_rows.append(fields)

conn.close()
compare_df = pd.DataFrame(compare_rows)[[
    "联系人", "个性签名(#4)", "地区(#5-7)", "标签(#30)", "朋友圈背景(#25)"
]]
compare_df


In [ ]:
conn = connect_ro(CONTACT_DB)
contacts = pd.read_sql_query(
    """
    SELECT username, remark, nick_name, alias, local_type, delete_flag,
           CASE WHEN username LIKE '%@chatroom' THEN 1 ELSE 0 END AS is_group
    FROM contact
    WHERE delete_flag = 0
    """,
    conn,
)
conn.close()

contacts["display_name"] = contacts.apply(
    lambda r: r["remark"] or r["nick_name"] or r["username"], axis=1
)

print(f"联系人总数: {len(contacts):,}  |  群聊: {contacts['is_group'].sum():,}  |  单聊/公众号等: {(~contacts['is_group'].astype(bool)).sum():,}")

contacts.sort_values("remark").tail(20)


In [ ]:
# 按关键词搜索联系人（修改 QUERY 后重新运行）
QUERY = "黄潋哲"

mask = (
    contacts["display_name"].str.contains(QUERY, case=False, na=False)
    | contacts["username"].str.contains(QUERY, case=False, na=False)
    | contacts["alias"].str.contains(QUERY, case=False, na=False)
)
contacts.loc[mask, ["username", "display_name", "alias", "is_group"]].head(30)


## 3. 会话列表 (session.db)

`SessionTable` 对应微信左侧聊天列表：`summary` 可能是 zstd 压缩字节，需解码。


In [ ]:
name_map = contacts.set_index("username")["display_name"].to_dict()

conn = connect_ro(SESSION_DB)
sessions_raw = pd.read_sql_query(
    """
    SELECT username, unread_count, summary, last_timestamp,
           last_msg_type, last_msg_sender, last_sender_display_name, is_hidden
    FROM SessionTable
    WHERE last_timestamp > 0
    ORDER BY last_timestamp DESC
    """,
    conn,
)
conn.close()


def decode_summary(val):
    if isinstance(val, bytes):
        text = decode_content(val, ct_flag=4)
    else:
        text = str(val or "")
    if ":\n" in text:
        text = text.split(":\n", 1)[1]
    return text[:120]


sessions = sessions_raw.copy()
sessions["display_name"] = sessions["username"].map(lambda u: name_map.get(u, u))
sessions["is_group"] = sessions["username"].str.contains("@chatroom", na=False)
sessions["last_time"] = pd.to_datetime(sessions["last_timestamp"], unit="s")
sessions["msg_type_name"] = sessions["last_msg_type"].map(format_msg_type)
sessions["summary_text"] = sessions["summary"].map(decode_summary)

cols = ["last_time", "display_name", "unread_count", "msg_type_name", "summary_text", "username"]
sessions[cols].head(25)


## 4. 聊天记录 (message/message_*.db)

每个聊天对象对应一张表 `Msg_<md5(username)>`，分布在多个 `message_N.db` 分片中。


### 4.0 聊天记录表结构说明

聊天记录存储在 `message/message_*.db` 中。每个聊天对象（wxid 或 `xxx@chatroom`）对应一张表：

```
Msg_<md5(username)>
```

例如 `wxid_friend_example` → `Msg_<md5>`。同一会话的消息可能分布在多个 `message_N.db` 分片里。

#### `Msg_*` 表字段（共 17 列）

| 字段 | 类型 | 含义 |
|------|------|------|
| **local_id** | INTEGER，主键 | 该聊天在本地的消息行 ID，自增；用于本地定位、语音转写、图片资源关联 |
| **server_id** | INTEGER | 微信服务器端消息 ID（MsgSvrID）；跨设备同步、撤回、去重 |
| **local_type** | INTEGER | 消息类型；**低 32 位**=主类型，**高 32 位**=子类型（见下表） |
| **sort_seq** | INTEGER | 排序序号（时间+毫秒后缀）；**按此字段排序 = 聊天时间顺序** |
| **real_sender_id** | INTEGER | 发送者在同库 `Name2Id` 表中的 **rowid**（不是 wxid 字符串） |
| **create_time** | INTEGER | Unix **秒级**时间戳 |
| **status** | INTEGER | 消息状态；常见 `3`（正常）、`4`（少量，多与撤回/异常相关） |
| **upload_status** | INTEGER | 媒体上传状态；文本消息多为 `0` |
| **download_status** | INTEGER | 媒体下载状态；`0`=无需/已完成，`1`/`2`=待下载或下载中 |
| **server_seq** | INTEGER | 服务器端序列号，用于同步排序 |
| **origin_source** | INTEGER | 消息来源；常见 `2`（正常聊天）、`10`（转发/引用类）、`0`（系统/特殊） |
| **source** | TEXT | 额外来源信息，多数为空 |
| **message_content** | TEXT/BLOB | **消息正文**（文本/XML/zstd 压缩二进制） |
| **compress_content** | TEXT | 备用压缩内容，很多聊天里为空 |
| **packed_info_data** | BLOB | 打包扩展元数据（protobuf/zstd），图片/文件类消息常见 |
| **WCDB_CT_message_content** | INTEGER | `message_content` 压缩类型：`0`=明文，`4`=zstd 需解压 |
| **WCDB_CT_source** | INTEGER | `source` 字段压缩类型，规则同上 |

#### `local_type` 主类型（低 32 位）

| 值 | 含义 |
|----|------|
| 1 | 文本 |
| 3 | 图片 |
| 34 | 语音 |
| 42 | 名片 |
| 43 | 视频 |
| 47 | 表情/贴纸 |
| 48 | 位置 |
| 49 | 链接 / 文件 / 小程序 / 公众号文章 |
| 50 | 语音/视频通话 |
| 10000 | 系统消息（如撤回通知） |
| 10002 | 系统通知 |

大数值示例：`21474836529` → 主类型 `1`（文本），子类型 `4`（高 32 位）。

```python
base_type = local_type & 0xFFFFFFFF
sub_type  = local_type >> 32
```

#### `message_content` 怎么读

- **文本**：直接字符串；群聊常见 `wxid_xxx:\n实际内容`
- **图片/语音/链接**：多为 XML（`<msg>...</msg>`），需解析 tag
- **压缩**：`WCDB_CT_message_content == 4` 时内容为 zstd 压缩 bytes，需先解压

#### 同库辅助表（`message_*.db`）

| 表名 | 作用 |
|------|------|
| **Name2Id** | wxid ↔ 数字 ID；`rowid` 对应 `real_sender_id` |
| **SendInfo** | 发送队列（chat_name_id + msg_local_id） |
| **DeleteInfo / DeleteResInfo** | 删除记录、被删资源路径 |
| **HistoryAddMsgInfo / HistorySysMsgInfo** | 历史同步、撤回标记 |
| **TimeStamp** | 库更新时间戳 |

#### 媒体资源库（`message_resource.db`）

| 表名 | 作用 |
|------|------|
| **ChatName2Id** | 聊天 username ↔ chat_id |
| **SenderName2Id** | 发送者 username ↔ sender_id |
| **MessageResourceInfo** | 消息与资源关联（`packed_info` 含图片 MD5） |
| **MessageResourceDetail** | 资源详情（大小、类型、路径索引） |

图片解密链路：`Msg_*.local_id` → `MessageResourceInfo.packed_info`（MD5）→ 磁盘 `.dat` 文件。

#### 治理/分析常用字段

1. **sort_seq** — 排序
2. **create_time** — 显示时间
3. **local_type** — 判断消息种类
4. **message_content + WCDB_CT_message_content** — 解码正文
5. **real_sender_id + Name2Id** — 群聊识别发送者
6. **local_id / server_id** — 去重、关联媒体、增量导出

下方 **§4.0 示例** 以与 **示例好友A**（`wxid_friend_example`）的单聊为例，展示真实数据。


In [ ]:
message_dbs = sorted(
    p for p in MESSAGE_DIR.glob("message_*.db")
    if not p.name.endswith(("_fts.db", "_resource.db"))
)

shard_stats = []
for db_path in message_dbs:
    conn = connect_ro(db_path)
    msg_tables = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name LIKE 'Msg_%'"
    ).fetchall()
    shard_stats.append({
        "数据库": db_path.name,
        "Msg_* 表数量": len(msg_tables),
        "大小(MB)": round(db_path.stat().st_size / 1024 / 1024, 1),
    })
    conn.close()

pd.DataFrame(shard_stats)


In [ ]:
def find_msg_table(username: str):
    """在所有 message_N.db 中定位聊天表，返回 (db_path, table_name)。"""
    table = username_to_msg_table(username)
    for db_path in message_dbs:
        conn = connect_ro(db_path)
        exists = conn.execute(
            "SELECT 1 FROM sqlite_master WHERE type='table' AND name=?",
            (table,),
        ).fetchone()
        conn.close()
        if exists:
            return db_path, table
    return None, None


def load_chat_messages(username: str, limit: int = 50, offset: int = 0) -> pd.DataFrame:
    db_path, table = find_msg_table(username)
    if not table:
        return pd.DataFrame()

    conn = connect_ro(db_path)
    sender_map = {r[0]: r[1] for r in conn.execute("SELECT rowid, user_name FROM Name2Id")}

    rows = conn.execute(
        f"""
        SELECT local_id, server_id, local_type, sort_seq, real_sender_id,
               create_time, status, message_content, WCDB_CT_message_content
        FROM [{table}]
        ORDER BY sort_seq DESC
        LIMIT ? OFFSET ?
        """,
        (limit, offset),
    ).fetchall()
    conn.close()

    records = []
    for r in rows:
        local_id, server_id, local_type, sort_seq, sender_id, create_time, status, raw, ct = r
        content = decode_content(raw, ct or 0)
        sender = sender_map.get(sender_id, "")
        records.append({
            "time": datetime.fromtimestamp(create_time),
            "type": format_msg_type(local_type),
            "sender": name_map.get(sender, sender or "我"),
            "content": content[:200] + ("..." if len(content) > 200 else ""),
            "local_id": local_id,
            "server_id": server_id,
        })
    return pd.DataFrame(records)


def _norm_name(s: str) -> str:
    """统一全角/半角括号等，便于备注名搜索。"""
    return (s or "").lower().replace("（", "(").replace("）", ")")


def resolve_username(query: str) -> str | None:
    q = _norm_name(query.strip())
    for _, row in contacts.iterrows():
        if q == _norm_name(row["username"]):
            return row["username"]
    for _, row in contacts.iterrows():
        for field in (row["display_name"], row["remark"], row["nick_name"]):
            n = _norm_name(field)
            if q == n or (q and q in n):
                return row["username"]
    return None


#### 4.0 实际示例：与 **示例好友A**（`wxid_friend_example`）的单聊

以下三个单元格依次展示：
1. **表结构**（17 个字段的 PRAGMA）
2. **最近 8 条消息**的完整字段 + 解码对话
3. **字段取值分布**（status、origin_source、local_type 等）

运行后你会看到类似这样的解码对话：

```
[2026-05-30 20:54:46]  我  ·  1 (文本)
  想象着，没我的日子

[2026-05-30 20:54:52]  我  ·  1 (文本)
  妙啊
```


In [ ]:
# 1) 表结构 + 基本信息
EXAMPLE_USERNAME = "wxid_friend_example"   # 联系人备注 示例好友A，260 条消息
EXAMPLE_SAMPLE_N = 8            # 取最近 N 条做字段对照

db_path, table_name = find_msg_table(EXAMPLE_USERNAME)
if not table_name:
    raise RuntimeError(f"未找到 {EXAMPLE_USERNAME} 的消息表")

schema_df = pd.read_sql_query(f"PRAGMA table_info([{table_name}])", connect_ro(db_path))
schema_df = schema_df.rename(columns={"name": "字段", "type": "类型", "pk": "主键"})
print(f"消息表: {table_name}")
print(f"所在分片: {db_path.name}")
print(f"联系人显示名: {name_map.get(EXAMPLE_USERNAME, EXAMPLE_USERNAME)}")
total = connect_ro(db_path).execute(f"SELECT COUNT(*) FROM [{table_name}]").fetchone()[0]
print(f"消息总数: {total:,}\n")
schema_df[["字段", "类型", "主键"]]


In [ ]:
# 3) Name2Id 映射 + 最近 N 条消息逐字段展示
conn = connect_ro(db_path)
name2id = pd.read_sql_query(
    "SELECT rowid AS real_sender_id, user_name, is_session FROM Name2Id LIMIT 10",
    conn,
)
print("Name2Id 样例（real_sender_id → user_name）：")
ipy_display(name2id)

sender_map = {r[0]: r[1] for r in conn.execute("SELECT rowid, user_name FROM Name2Id")}

rows = conn.execute(
    f"""
    SELECT local_id, server_id, local_type, sort_seq, real_sender_id,
           create_time, status, upload_status, download_status,
           server_seq, origin_source, source,
           message_content, compress_content, packed_info_data,
           WCDB_CT_message_content, WCDB_CT_source
    FROM [{table_name}]
    ORDER BY sort_seq DESC
    LIMIT ?
    """,
    (EXAMPLE_SAMPLE_N,),
).fetchall()

records = []
for r in reversed(rows):
    (local_id, server_id, local_type, sort_seq, real_sender_id,
     create_time, status, upload_status, download_status,
     server_seq, origin_source, source,
     raw_content, compress_content, packed_info, ct_content, ct_source) = r

    raw_text = decode_content(raw_content, ct_content or 0)
    base_type, sub_type = split_msg_type(local_type)
    sender_u = sender_map.get(real_sender_id, "")
    who = name_map.get(EXAMPLE_USERNAME, EXAMPLE_USERNAME) if sender_u == EXAMPLE_USERNAME else "我"

    records.append({
        "时间": datetime.fromtimestamp(create_time).strftime("%Y-%m-%d %H:%M:%S"),
        "发送者": who,
        "主类型": f"{base_type} ({format_msg_type(local_type)})",
        "子类型": sub_type,
        "解码正文": friendly_content(local_type, raw_text, EXAMPLE_USERNAME, sender_u),
        "local_id": local_id,
        "server_id": str(server_id),
        "sort_seq": sort_seq,
        "real_sender_id": real_sender_id,
        "sender_wxid": sender_u,
        "status": status,
        "upload_status": upload_status,
        "download_status": download_status,
        "origin_source": origin_source,
        "WCDB_CT_content": ct_content,
        "packed_info(B)": len(packed_info) if packed_info else 0,
        "原始正文预览": (raw_text[:120] + "…") if len(raw_text) > 120 else raw_text,
    })

conn.close()
example_df = pd.DataFrame(records)

print("=" * 72)
print("对话形式（解码后）")
print("=" * 72)
for _, row in example_df.iterrows():
    print(f"\n[{row['时间']}]  {row['发送者']}  ·  {row['主类型']}")
    print(f"  {row['解码正文']}")
    if row["原始正文预览"] != row["解码正文"]:
        print(f"  └─ 原始: {row['原始正文预览']}")
print("\n" + "=" * 72)
example_df


In [ ]:
# 4) 各字段取值分布（帮助理解 status / origin_source / WCDB_CT 等）
conn = connect_ro(db_path)

def field_distribution(col: str, top: int = 8) -> pd.DataFrame:
    df = pd.read_sql_query(
        f"SELECT {col} AS 值, COUNT(*) AS 条数 FROM [{table_name}] GROUP BY {col} ORDER BY 条数 DESC LIMIT {top}",
        conn,
    )
    df["占比"] = (df["条数"] / total * 100).round(1).astype(str) + "%"
    return df

print(f"字段分布（{name_map.get(EXAMPLE_USERNAME, EXAMPLE_USERNAME)}，共 {total:,} 条）\n")
for col in ["status", "download_status", "origin_source", "WCDB_CT_message_content"]:
    print(f"--- {col} ---")
    ipy_display(field_distribution(col))

print("--- local_type 主类型（低32位）---")
type_dist = pd.read_sql_query(
    f"""
    SELECT (local_type & 4294967295) AS 主类型, COUNT(*) AS 条数
    FROM [{table_name}] GROUP BY 1 ORDER BY 条数 DESC
    """,
    conn,
)
type_dist["类型名"] = type_dist["主类型"].map(lambda t: MSG_TYPE_NAMES.get(int(t), f"type={t}"))
type_dist["占比"] = (type_dist["条数"] / total * 100).round(1).astype(str) + "%"
conn.close()
type_dist


In [ ]:
# 方式1: 按备注/昵称/wxid 搜索（留空则自动取最近活跃会话）
CHAT_QUERY = ""

# 方式2: 若已知 username，可直接指定（优先级高于 CHAT_QUERY）
USERNAME = ""  # 例如 "44961104992@chatroom"

if USERNAME:
    username = USERNAME
elif CHAT_QUERY:
    username = resolve_username(CHAT_QUERY) or CHAT_QUERY
else:
    username = sessions.iloc[0]["username"]

db_path, table = find_msg_table(username)
print(f"username: {username}")
print(f"显示名: {name_map.get(username, username)}")
print(f"消息表: {table} @ {db_path.name if db_path else '未找到'}")

if table:
    msgs = load_chat_messages(username, limit=30)
    msgs[["time", "type", "sender", "content"]]
else:
    print("未找到该聊天的消息表。")


## 4.1 展开聊天记录（完整可读内容）

下面以对话形式展示每条消息的**解码后正文**，并保留 `local_id`、原始 XML（链接/文件类）等字段供核对。


In [ ]:
def load_chat_messages_full(username: str, limit: int = 50, offset: int = 0, oldest_first: bool = True) -> pd.DataFrame:
    """加载完整消息（不截断 content）。"""
    db_path, table = find_msg_table(username)
    if not table:
        return pd.DataFrame()

    order = "ASC" if oldest_first else "DESC"
    conn = connect_ro(db_path)
    sender_map = {r[0]: r[1] for r in conn.execute("SELECT rowid, user_name FROM Name2Id")}
    is_group = username.endswith("@chatroom") or username.endswith("@openim")

    rows = conn.execute(
        f"""
        SELECT local_id, server_id, local_type, sort_seq, real_sender_id,
               create_time, status, message_content, WCDB_CT_message_content
        FROM [{table}]
        ORDER BY sort_seq {order}
        LIMIT ? OFFSET ?
        """,
        (limit, offset),
    ).fetchall()
    total = conn.execute(f"SELECT COUNT(*) FROM [{table}]").fetchone()[0]
    conn.close()

    records = []
    for r in rows:
        local_id, server_id, local_type, sort_seq, sender_id, create_time, status, raw, ct = r
        raw_text = decode_content(raw, ct or 0)
        sender_u = sender_map.get(sender_id, "")
        if is_group:
            who = name_map.get(sender_u, sender_u or "?")
        elif sender_u == username:
            who = name_map.get(username, username)
        else:
            who = "我"
        records.append({
            "time": datetime.fromtimestamp(create_time),
            "sender": who,
            "type": format_msg_type(local_type),
            "content": friendly_content(local_type, raw_text, username, sender_u),
            "raw_content": raw_text,
            "local_id": local_id,
            "server_id": str(server_id),
            "sort_seq": sort_seq,
        })
    df = pd.DataFrame(records)
    df.attrs["total"] = total
    df.attrs["username"] = username
    return df


def show_chat_transcript(df: pd.DataFrame, show_raw_for_types=("链接/文件", "系统", "撤回")):
    """以对话形式打印聊天记录。"""
    if df.empty:
        print("无消息")
        return
    username = df.attrs.get("username", "")
    title = name_map.get(username, username)
    total = df.attrs.get("total", len(df))
    print(f"=== 与 [{title}] 的对话 ===")
    print(f"username: {username}  |  库内共 {total:,} 条  |  本次展示 {len(df)} 条\n")
    print("=" * 70)
    for _, row in df.iterrows():
        ts = row["time"].strftime("%Y-%m-%d %H:%M:%S")
        print(f"\n[{ts}]  {row['sender']}  ·  {row['type']}  (id={row['local_id']})")
        print(row["content"])
        if row["type"] in show_raw_for_types and row["raw_content"] != row["content"]:
            preview = row["raw_content"][:600]
            suffix = "..." if len(row["raw_content"]) > 600 else ""
            print(f"  └─ 原始内容: {preview}{suffix}")
    print("\n" + "=" * 70)


In [ ]:
# 修改这里选择要展开的聊天（备注/昵称/wxid 均可）
DETAIL_CHAT_QUERY = "示例好友A"   # 例如 "示例好友A"、或留空用最近会话
DETAIL_USERNAME = ""           # 已知 username 时可直接填，如 "wxid_friend_example"
DETAIL_LIMIT = 30              # 展示条数
DETAIL_OFFSET = 0              # 分页偏移（0=最早的消息段；配合 oldest_first=False 可翻页看更新消息）

if DETAIL_USERNAME:
    detail_username = DETAIL_USERNAME
elif DETAIL_CHAT_QUERY:
    detail_username = resolve_username(DETAIL_CHAT_QUERY) or DETAIL_CHAT_QUERY
else:
    detail_username = sessions.iloc[0]["username"]

detail_df = load_chat_messages_full(
    detail_username,
    limit=DETAIL_LIMIT,
    offset=DETAIL_OFFSET,
    oldest_first=False,  # False = 取最近 N 条，再按时间正序展示
)
if not detail_df.empty and not DETAIL_OFFSET:
    # 最近 N 条：先 DESC 取出，再按时间正序排列便于阅读
    detail_df = detail_df.sort_values("time").reset_index(drop=True)

show_chat_transcript(detail_df)
detail_df[["time", "sender", "type", "content", "local_id"]]


## 4.2 引用消息（回复某条消息）是怎么存的

微信的「引用回复」**不是单独的关系表，也没有外键列**，而是把被引用消息的信息整段塞进**引用方那条消息的 `message_content` XML 内部**。

### 类型识别

- 引用消息的 `local_type` 低 32 位 = **49**（appmsg 大类，notebook 现在显示成「链接/文件」）。
- 其 `message_content`（zstd 压缩，`WCDB_CT_message_content=4`）解压后是 appmsg XML，其中 **`<appmsg><type>57</type>`** 专门表示「引用回复」。
- XML 里的 **`<refermsg>`** 块装着被引用消息的全部信息。

### `<refermsg>` 字段含义

| 字段 | 含义 |
|------|------|
| `<type>` | 被引用消息的类型（1=文本、3=图片…） |
| `<svrid>` | **被引用消息的 `server_id`** —— 这就是连接两条消息的唯一可靠的键 |
| `<fromusr>` / `<chatusr>` | 被引用消息的发送者 / 所属会话 |
| `<displayname>` | 被引用方昵称 |
| `<content>` | 被引用消息的正文（**原文拷贝快照**） |
| `<createtime>` | 被引用消息的发送时间 |

### 关键结论

1. **关联指针 = `<refermsg><svrid>`**，它的值**正好等于被引用消息的 `server_id`**。顺着它就能在同一张 `Msg_*` 表里 `WHERE server_id = <svrid>` 找到原消息。
2. 引用是**自包含**的：`<refermsg>` 里冗余存了被引用消息的 `content`/`displayname`/`createtime`，即使原消息被删，引用里仍有当时的文本快照。
3. 引用方自己说的话在 appmsg 的 **`<title>`** 里（不是 `<content>`）。

### 下面用「示例好友A」里的真实例子验证：local_id=257（引用方）→ local_id=255（被引用）。

被引用消息（local_id=255，示例好友A发的文本）：
```text
local_id     = 255
server_id    = 7338366730403543568   ← 关键
local_type   = 1 (文本)
message_content = "我是不是来到了你的城市"
```
引用方消息（local_id=257，你发的）：
```text
local_id   = 257
local_type = 244813135921  → 低32位 = 49（链接/文件），高位是扩展标志
WCDB_CT_message_content = 4  → message_content 是 zstd 压缩的
```
它的 `message_content` 解压后是一段 appmsg XML，核心就是 `<type>57</type>` + `<refermsg>` 块：
```xml
<appmsg>
    <title>熟悉的那一条街</title>       ← 你这条引用消息本身的正文
    <type>57</type>                      ← 57 = 引用回复类型
    <refermsg>
        <type>1</type>                   ← 被引用消息的类型（1=文本）
        <svrid>7338366730403543568</svrid>  ← 被引用消息的 server_id
        <fromusr>wxid_friend_example</fromusr>
        <chatusr>wxid_friend_example</chatusr>
        <displayname>洵淼</displayname>     ← 被引用方昵称
        <content>我是不是来到了你的城市</content>  ← 被引用内容（原文拷贝）
        <createtime>1780145664</createtime>
    </refermsg>
</appmsg>
```


In [ ]:
import re

# ===== 引用消息实例分析：示例好友A =====
REF_USERNAME = resolve_username("示例好友A") or "wxid_friend_example"
db_path, table = find_msg_table(REF_USERNAME)
print(f"会话: {REF_USERNAME}  |  表: {table}\n")


def fetch_msg(local_id: int):
    conn = connect_ro(db_path)
    row = conn.execute(
        f"SELECT local_id, server_id, local_type, create_time, "
        f"message_content, WCDB_CT_message_content FROM [{table}] WHERE local_id=?",
        (local_id,),
    ).fetchone()
    conn.close()
    if not row:
        return None
    return {
        "local_id": row[0], "server_id": row[1], "local_type": row[2],
        "create_time": row[3],
        "content": decode_content(row[4], row[5] or 0),
    }


# --- 引用方消息（你发的，引用了对方的话）---
quoting = fetch_msg(257)
base_type, _ = split_msg_type(quoting["local_type"])
print("【引用方消息 local_id=257】")
print(f"  local_type 原值={quoting['local_type']}  低32位={base_type}（49=appmsg）")
print(f"  server_id={quoting['server_id']}")
print(f"  解码后 message_content XML：\n{quoting['content']}\n")

# --- 从 XML 解析 refermsg ---
xml = quoting["content"]
appmsg_type = re.search(r"<appmsg>.*?<type>(\d+)</type>", xml, re.DOTALL)
refer = re.search(r"<refermsg>(.*?)</refermsg>", xml, re.DOTALL)
print("=" * 60)
print(f"appmsg/type = {appmsg_type.group(1) if appmsg_type else '?'}  (57=引用回复)")

def pick(tag, s):
    m = re.search(rf"<{tag}>(.*?)</{tag}>", s, re.DOTALL)
    return m.group(1).strip() if m else ""

my_words = pick("title", xml)
print(f"我实际发的话(<title>) = {my_words!r}")

if refer:
    rb = refer.group(1)
    refer_svrid = pick("svrid", rb)
    print("\n--- <refermsg> 解析（被引用消息信息）---")
    print(f"  svrid       = {refer_svrid}   ← 指向被引用消息的 server_id")
    print(f"  type        = {pick('type', rb)}")
    print(f"  displayname = {pick('displayname', rb)}")
    print(f"  content     = {pick('content', rb)!r}")
    print(f"  createtime  = {pick('createtime', rb)}")

    # --- 顺着 svrid 回查被引用的原消息 ---
    print("\n" + "=" * 60)
    print("顺着 svrid 在同表中 WHERE server_id = svrid 回查原消息：")
    quoted = fetch_msg(255)  # 已知 local_id=255，用它的 server_id 对照
    print(f"  local_id=255 的 server_id = {quoted['server_id']}")
    print(f"  refermsg.svrid            = {refer_svrid}")
    print(f"  >>> 两者相等? {str(quoted['server_id']) == refer_svrid}")
    print(f"  被引用原文(local_id=255)  = {quoted['content']!r}")


## 4.3 `message` 文件夹下的各类库（`message_*` vs `biz_message_*`）

`decrypted/message/` 下其实有**几类不同用途**的库，命名前缀区分：

| 文件 | 类别 | 装什么 | 是否纳入治理库 |
|------|------|--------|----------------|
| `message_<N>.db` | **普通聊天** | 与**人 / 群 / 企业微信**的真实往来消息 | ✅ 纳入 `messages` |
| `biz_message_<N>.db` | **公众号/服务号推送** | 关注的**公众号文章推送、服务号通知**（账号都是 `gh_*` / `@app` / `brandprivatemsg`）| ❌ 不纳入（决策 2） |
| `media_0.db` | 媒体（语音） | 语音 silk 数据，按 `server_id` 回查 | ❌ 仅回查 |
| `message_resource.db` | 媒体资源索引 | 图片/视频/文件的资源映射 | ❌ 仅回查 |
| `message_fts.db` | 全文搜索索引 | 微信自带的搜索倒排索引 | ❌ 丢弃 |

### `message_<N>.db` vs `biz_message_<N>.db` 的关系

- **表结构完全一样**：两者都是一堆 `Msg_<md5(username)>` 表 + `Name2Id` + `TimeStamp` 等，
  字段也都是 `local_id / server_id / local_type / real_sender_id / message_content / ...`（17 列）。
- **区别只在「会话对象是谁」**：
  - `message_*` 的会话对象是 `wxid_*`（好友）、`*@chatroom`（群）、`*@openim`（企业微信）；
  - `biz_message_*` 的会话对象是 `gh_*`（公众号）、`*@app`（服务/小程序）、`brandprivatemsg@hardcode`（品牌私信）。
- **都按时间分片**：同一个账号的消息也会跨多个 `*_<N>.db` 分片，需遍历全部分片并用 `server_id` 去重。

### 为什么治理库不纳入 `biz_message`

公众号推送是**单向营销/资讯内容**（文章标题、链接、服务通知），不是你与人的真实对话，
对"模仿你说话/还原社交关系"的训练目标是噪声，故按 §6 决策 2 **整体排除**。
若将来要做"我关注了哪些公众号/读了什么"类分析，可单独再建一张表，不污染 `messages`。

下面用代码对比两类库的规模与账号类型。


In [ ]:
import re as _re

def _shards(pattern: str):
    """只匹配 prefix_<数字>.db，排除 fts/resource 等。"""
    out = []
    for p in MESSAGE_DIR.glob(pattern):
        if _re.fullmatch(pattern.replace("*", r"\d+"), p.name):
            out.append(p)
    return sorted(out)


def _scan_category(shards):
    """统计一类库：分片数、会话表数、消息总数、账号类型分布。"""
    from collections import Counter
    n_tables = n_msgs = 0
    kinds = Counter()
    for db in shards:
        conn = connect_ro(db)
        names = {rid: u for rid, u in conn.execute("SELECT rowid, user_name FROM Name2Id")}
        tabs = [r[0] for r in conn.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name LIKE 'Msg\\_%' ESCAPE '\\'")]
        n_tables += len(tabs)
        for t in tabs:
            n_msgs += conn.execute(f"SELECT COUNT(*) FROM [{t}]").fetchone()[0]
        for u in names.values():
            if not u:
                continue
            if u.endswith("@chatroom"):
                kinds["群聊 @chatroom"] += 1
            elif u.endswith("@openim"):
                kinds["企业微信 @openim"] += 1
            elif u.endswith("@app"):
                kinds["服务/小程序 @app"] += 1
            elif u.startswith("gh_"):
                kinds["公众号 gh_"] += 1
            elif u.startswith("wxid_"):
                kinds["个人 wxid_"] += 1
            else:
                kinds["其他"] += 1
        conn.close()
    return n_tables, n_msgs, kinds


for label, pat in [("message_*（普通聊天）", "message_*.db"),
                   ("biz_message_*（公众号推送）", "biz_message_*.db")]:
    shards = _shards(pat)
    n_tables, n_msgs, kinds = _scan_category(shards)
    print(f"=== {label} ===")
    print(f"  分片: {len(shards)} 个 | 会话表: {n_tables} | 消息: {n_msgs:,}")
    print(f"  账号类型分布: {dict(kinds)}\n")


## 5. 单表原始数据预览

修改 `DB_REL` 和 `TABLE_NAME` 可直接查看任意表的 schema 与样例行。


In [ ]:
DB_REL = "message/message_resource.db"   # 例如 contact/contact.db, sns/sns.db
TABLE_NAME = "ChatName2Id"               # 例如 contact, SessionTable, MessageResourceInfo

db_path = DECRYPTED_DIR / DB_REL
conn = connect_ro(db_path)

schema = pd.read_sql_query(f"PRAGMA table_info([{TABLE_NAME}])", conn)
print(f"库: {DB_REL}  |  表: {TABLE_NAME}")
schema

sample = pd.read_sql_query(f"SELECT * FROM [{TABLE_NAME}] LIMIT 10", conn)
conn.close()
sample


## 6. 其他数据库快速索引

非 message 库通常表较少，可直接列出全部表结构。


In [ ]:
OTHER_DBS = [
    "sns/sns.db",
    "favorite/favorite.db",
    "general/general.db",
    "emoticon/emoticon.db",
    "bizchat/bizchat.db",
]

for rel in OTHER_DBS:
    path = DECRYPTED_DIR / rel
    if not path.exists():
        print(f"[跳过] {rel} 不存在")
        continue
    df = list_tables(path, skip_msg_tables=False)
    print(f"\n=== {rel} ({path.stat().st_size/1024/1024:.1f} MB) ===")
    df


---

**下一步建议：** 确定要治理的数据范围（联系人 / 会话 / 消息类型）后，可在此基础上做清洗规则、导出规范或质量检测。


## 7. 治理库 `governed.db` 产出验证

由 `build_governed_db.py` 生成的治理库：**4 张业务表** + 构建内部 `build_meta`：
`contacts` / `chatrooms` / `chatroom_members` / `messages` / `build_meta`。

v2 双账号（`account` = `personal` / `work`）；messages 去重后约 **238 万**（源扫描 ~310 万，跨分片去重跳过 ~73 万）。

本节验证产出质量：
1. **行数、按账号分布与消息类型**；
2. **联系人 + 聊天记录抽样**（默认 `account='personal'`，引用 JOIN 带 `account`）；
3. **媒体回查 demo**：从 `messages` 的 `server_id` 回到 personal 解密库取语音/图片原始数据。


In [ ]:
import sqlite3
from pathlib import Path

GOVERNED_DB = Path("../output/governed.db")
assert GOVERNED_DB.exists(), f"未找到 {GOVERNED_DB}，请先运行 build_governed_db.py"

gconn = sqlite3.connect(f"file:{GOVERNED_DB}?mode=ro", uri=True)
gconn.row_factory = sqlite3.Row

print("=== 7.1 各表行数 ===")
for t in ("contacts", "chatrooms", "chatroom_members", "messages", "build_meta"):
    n = gconn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:18} {n:>12,}")

print("\n=== messages 按 account ===")
for r in gconn.execute("SELECT account, COUNT(*) c FROM messages GROUP BY account ORDER BY account"):
    print(f"  {r['account']:10} {r['c']:>12,}")

print("\n=== 消息类型分布（personal） ===")
for r in gconn.execute(
    "SELECT type, COUNT(*) c FROM messages WHERE account='personal' GROUP BY type ORDER BY c DESC"
):
    print(f"  {r['type']:12} {r['c']:>12,}")

print("\n=== 自己 vs 他人 ===")
for r in gconn.execute("SELECT is_self, COUNT(*) c FROM messages GROUP BY is_self"):
    print(f"  {'自己发送' if r['is_self'] else '他人发送'}: {r['c']:,}")

print("\n=== 引用消息 reply_quote 兜底情况 ===")
q_total = gconn.execute("SELECT COUNT(*) FROM messages WHERE type='quote'").fetchone()[0]
q_kept = gconn.execute("SELECT COUNT(*) FROM messages WHERE type='quote' AND reply_quote IS NOT NULL").fetchone()[0]
print(f"  引用消息共 {q_total:,} 条；原文在库内 {q_total - q_kept:,} 条（reply_to 自关联），"
      f"原文缺失保留快照 {q_kept:,} 条")


In [ ]:
from datetime import datetime

# === 7.2 联系人 + 聊天记录抽样：示例好友A（personal 号）===
SAMPLE_ACCOUNT = "personal"
SAMPLE_USER = "wxid_friend_example"

c = gconn.execute(
    "SELECT * FROM contacts WHERE account=? AND username=?", (SAMPLE_ACCOUNT, SAMPLE_USER)
).fetchone()
print("=== contacts 行 ===")
for k in ("username", "type", "is_friend", "nick_name", "remark", "gender",
          "country", "province", "city", "labels", "signature"):
    print(f"  {k:12} = {c[k]!r}")

print("\n=== 该会话消息抽样（按时间，含引用自关联）===")
rows = gconn.execute("""
    SELECT q.timestamp, q.is_self, q.sender_username, q.type, q.content,
           q.reply_to, o.content AS quoted_content, o.sender_username AS quoted_from
    FROM messages q
    LEFT JOIN messages o
      ON o.account = q.account AND o.server_id = q.reply_to AND o.server_id != 0
    WHERE q.account = ? AND q.chat_username = ?
    ORDER BY q.timestamp
    LIMIT 40
""", (SAMPLE_ACCOUNT, SAMPLE_USER)).fetchall()

for r in rows:
    ts = datetime.fromtimestamp(r["timestamp"]).strftime("%m-%d %H:%M")
    who = "我" if r["is_self"] else (r["sender_username"] or "?")
    line = f"[{ts}] {who:>4} ·{r['type']}· {str(r['content'])[:40]}"
    if r["reply_to"]:
        if r["quoted_content"] is not None:
            line += f"   ↩引用[{r['quoted_from']}]: {str(r['quoted_content'])[:25]}"
        else:
            line += "   ↩引用[原文不在库，见 reply_quote]"
    print(line)


In [ ]:
import json

# === 7.3 媒体回查 demo：从 messages.server_id 回到原始库取原始数据 ===
MEDIA_DB = DECRYPTED_DIR / "message" / "media_0.db"
RESOURCE_DB = DECRYPTED_DIR / "message" / "message_resource.db"


def lookup_voice(server_id: int):
    """语音：media_0.db / VoiceInfo.svr_id == server_id → voice_data (silk BLOB)。"""
    conn = connect_ro(MEDIA_DB)
    conn.row_factory = sqlite3.Row  # 允许用列名访问（connect_ro 默认返回元组）
    r = conn.execute(
        "SELECT svr_id, local_id, length(voice_data) AS nbytes FROM VoiceInfo WHERE svr_id=?",
        (server_id,)).fetchone()
    conn.close()
    return r


def lookup_resource(server_id: int):
    """图片/视频/文件：message_resource.db / MessageResourceInfo.message_svr_id == server_id。"""
    conn = connect_ro(RESOURCE_DB)
    conn.row_factory = sqlite3.Row  # 允许用列名访问
    info = conn.execute(
        "SELECT message_id, message_svr_id, message_local_type FROM MessageResourceInfo "
        "WHERE message_svr_id=?", (server_id,)).fetchone()
    details = []
    if info:
        details = conn.execute(
            "SELECT resource_id, type, size FROM MessageResourceDetail WHERE message_id=?",
            (info[0],)).fetchall()
    conn.close()
    return info, details


# --- 语音回查 ---
print("=== 语音回查 ===")
v = gconn.execute(
    "SELECT server_id, media_ref FROM messages "
    "WHERE account='personal' AND type='voice' AND server_id!=0 LIMIT 1"
).fetchone()
print(f"  治理库 messages: server_id={v['server_id']}")
print(f"    media_ref={v['media_ref']}")
vr = lookup_voice(v["server_id"])
if vr:
    print(f"  → media_0.db/VoiceInfo: svr_id={vr['svr_id']} local_id={vr['local_id']} "
          f"voice_data={vr['nbytes']:,} 字节（silk 格式，可解码为音频）")

# --- 图片回查 ---
print("\n=== 图片回查 ===")
im = gconn.execute(
    "SELECT server_id, media_ref FROM messages "
    "WHERE account='personal' AND type='image' AND server_id!=0 LIMIT 1"
).fetchone()
print(f"  治理库 messages: server_id={im['server_id']}")
mref = json.loads(im["media_ref"])
print(f"    media_ref.md5={mref.get('md5')}  length={mref.get('length')}")
info, details = lookup_resource(im["server_id"])
if info:
    print(f"  → message_resource.db/MessageResourceInfo: message_id={info[0]} "
          f"local_type={info[2]}")
    for d in details:
        print(f"      MessageResourceDetail: resource_id={d[0]} type={d[1]} size={d[2]:,} 字节")
    print("  （再按 md5 去磁盘 xwechat_files/.../Img/<md5>.dat 取密文，用 decode_image.py 解密即得原图）")

print("\n✅ 验证完成：治理库行数/类型正常，引用自关联可用，媒体可凭 server_id 回查原始库。")


### 7.4 对照：方案 A —— `with closing(...)` 自管理连接

上面 §7.1–7.3 用的是**方案 B**：复用一个全局 `gconn`，分析全部做完后在 §8 统一关闭。

下面演示**方案 A**：每次查询**自己开自己的连接，用完即自动关闭**，无需任何全局变量、也不必担心
「忘了 close」或「重复运行报连接已关」。两种风格产出完全一致，按场景择一即可：

| | 方案 B（全局 `gconn`） | 方案 A（`with closing`） |
|---|------------------------|--------------------------|
| 适合 | 交互式探索、多格反复跑同一连接 | 一次性脚本、函数封装、即用即弃 |
| 关闭 | 末尾手动 `gconn.close()` | 出 `with` 块**自动**关闭 |
| 重复运行 | 需注意连接是否已关 | 天然安全，每次全新连接 |

> `contextlib.closing(obj)` 会在退出 `with` 块时自动调用 `obj.close()`。
> （直接 `with sqlite3.connect(...)` 只会管事务提交/回滚，**不会** close 连接，所以这里用 `closing` 包一层。）


In [ ]:
from contextlib import closing


def gquery(sql: str, params: tuple = ()):
    """方案 A：每次调用自开自关一个只读连接，用完即释放。"""
    with closing(sqlite3.connect(f"file:{GOVERNED_DB}?mode=ro", uri=True)) as conn:
        conn.row_factory = sqlite3.Row
        return conn.execute(sql, params).fetchall()
    # 出了 with 块，连接已自动 close，无需手动管理


# --- 用方案 A 重做 §7.1 的行数统计（产出与上面一致）---
print("=== 各表行数（方案 A：with closing 自管理连接）===")
for t in ("contacts", "chatrooms", "chatroom_members", "messages", "build_meta"):
    n = gquery(f"SELECT COUNT(*) AS c FROM {t}")[0]["c"]
    print(f"  {t:18} {n:>12,}")

# --- 用方案 A 抽样一条引用消息，验证自关联可用 ---
print("\n=== 引用消息抽样（方案 A，personal）===")
rows = gquery("""
    SELECT q.content AS my_reply,
           COALESCE(o.content, q.reply_quote) AS quoted
    FROM messages q
    LEFT JOIN messages o
      ON o.account = q.account AND o.server_id = q.reply_to AND o.server_id != 0
    WHERE q.account = 'personal' AND q.type = 'quote'
    LIMIT 3
""")
for r in rows:
    print(f"  我: {str(r['my_reply'])[:24]!r}  ↩ 被引用: {str(r['quoted'])[:30]!r}")

print("\n说明：本格全程没有用到全局 gconn，每次查询的连接都已在 with 块结束时自动关闭。")


## 8. 收尾：释放数据库连接

§7 全程复用同一个全局只读连接 `gconn`（不在分析单元格里 `close`，以便各单元格可反复运行）。
**所有分析做完后**，运行下面这一格手动关闭连接、释放文件句柄。

> 即使忘了运行也无大碍：`gconn` 是只读连接，不会锁库；关闭 Notebook 内核时句柄会被系统回收。


In [ ]:
# 收尾：关闭全局只读连接（可重复运行，已关闭不会报错）
try:
    gconn.close()
    print("✅ gconn 已关闭，连接释放。")
except NameError:
    print("gconn 尚未创建（请先运行 §7.1）。")
except Exception as e:
    print(f"关闭时忽略异常：{e}")
